# Imports

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp, input_file_name
from pyspark.sql import functions as F


## Project Configuration

In [0]:
catalog = "crypto_exchange"

landing_schema = "landing"
bronze_schema = "bronze"

volume = "cryptoexchange_data"

base_path = f"/Volumes/{catalog}/{landing_schema}/{volume}"

datasets = [
    "conversions",
    "crypto_list",
    "market_data"
]

## Generic Bronze AutoLoader Function

In [0]:
def create_bronze_table(dataset):
    input_path = f"{base_path}/{dataset}"

    @dp.table(
        name = f"{bronze_schema}_{dataset}"
    )

    def bronze_table():
        return(
            spark.readStream
            .format("cloudFiles")

            .option(
                "cloudFiles.format",
                "json"
            )
            .option("multiline", "true")
            
            .option(
                "cloudFiles.schemaEvolutionMode",
                "addNewColumns"
            )
            .option(
                "cloudFiles.rescuedDataColumn",
                "_rescued_data_"
            )
            .load(input_path)
            .select(
                "*",
                F.col("_metadata.file_path").alias("input_file_path"),
                F.current_timestamp().alias("ingest_timestamp")
            )
        )


## Ingesting the data

In [0]:
for dataset in datasets:
    create_bronze_table(dataset)
